In [6]:
# Step 1: rf_model.ipynb
# Build Random Forest IDS Model
# Load preprocessed CICIDS2017 data

import numpy as np # np = numpy (use loading .npy files, fast number calculations; for Math, arrays)
import pandas as pd # pd = pandas (use Tables, csv files, work with dataframes)
print("Loading preprocessed data...")

# Load saved fles 
save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/CICIDS-2017/processed/"
X_train = np.load(save_path + "X_train.npy") # np.load() 
X_test = np.load(save_path + "X_test.npy")
y_train = pd.read_csv(save_path + "Y_train.csv").squeeze() # squeeze() = remove extra dimensions, this model need 1 dimension
y_test = pd.read_csv(save_path + "y_test.csv").squeeze()

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")
print("Data  loaded!")

Loading preprocessed data...
X_train: (1979513, 80)
X_test: (848363, 80)
y_train: (1979513,)
y_test: (848363,)
Data  loaded!


In [9]:
# Step 2: Build and train Random Forest model
# RF = many decision trees voting together
# like asking 100 security experts majority vote

from sklearn.ensemble import RandomForestClassifier
import time

print("Building Random Forest model...")
print("This may take 5-10 minutes on the device...")

# Create RF model
# n_estimators = number of trees(100 is standard)
# random_state = reproducible results
# n_jobs = -1 means use all CPU cores
rf_model = RandomForestClassifier(
    n_estimators = 100,
    random_state = 42,
    n_jobs = -1,
    verbose = 1
)

# Train model - learn from training data
start_time = time.time()
rf_model.fit(X_train, y_train)
end_time = time.time()

training_time = round(end_time - start_time, 2)
print(f"Training complete")
print(f"Training time: {training_time} seconds")

Building Random Forest model...
This may take 5-10 minutes on the device...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:  9.4min


Training complete
Training time: 1245.48 seconds


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed: 20.5min finished


In [11]:
# Step 3: Evaluate Random Forest model
# Measure Acc, P, R, F1 on test data
from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import time

print("Evaluating model on test data...")
print("Make prodictions...")
start_time = time.time()
y_pred = rf_model.predict(X_test)
end_time = time.time()

prediction_time = round(end_time - start_time, 2)
# Calculate  metrics 
acc = accuracy_score(y_test, y_pred)
p = precision_score(y_test, y_pred,
                   average ='weighted')
r = recall_score(y_test, y_pred,
                average='weighted')
f1 = f1_score(y_test,y_pred,
             average = 'weighted')

print(f"\n== Random Forest Results ===")
print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {p:.4f} ({p*100:.2f}%)")
print(f"Recall:    {r:.4f} ({r*100:.2f}%)")
print(f"F1_Score:  {f1:.4f} ({f1*100:.2f}%)")
print(f"Prediction time: {prediction_time}s")
print(f"\nDetailed Report:")
print(classification_report(y_test, y_pred))


# Infiltration attack:
# = attacker sneaks inside network disguised as legitimate traffic
# = most dangerous but rarest attack
# = only 36 training samples → hard to learn
# = RF F1 = 78% (lowest of all attacks)
# = class imbalance problem
# KEY FINDING: rare attacks need more data
#              XAI helps identify why model misses them 

Evaluating model on test data...
Make prodictions...


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    4.2s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    8.5s finished



== Random Forest Results ===
Accuracy:  0.9994 (99.94%)
Precision: 0.9994 (99.94%)
Recall:    0.9994 (99.94%)
F1_Score:  0.9994 (99.94%)
Prediction time: 13.16s

Detailed Report:
                  precision    recall  f1-score   support

          BENIGN       1.00      1.00      1.00    681396
             Bot       0.97      0.95      0.96       587
            DDoS       1.00      1.00      1.00     38408
   DoS GoldenEye       1.00      0.99      1.00      3088
        DoS Hulk       1.00      1.00      1.00     69037
DoS Slowhttptest       0.99      0.99      0.99      1650
   DoS slowloris       1.00      1.00      1.00      1739
     FTP-Patator       1.00      1.00      1.00      2380
      Heartbleed       1.00      1.00      1.00         3
    Infiltration       1.00      0.64      0.78        11
        PortScan       1.00      1.00      1.00     47641
     SSH-Patator       1.00      1.00      1.00      1769
      Web Attack       0.99      0.97      0.98       654

      

In [14]:
# Step 4: Save RF model
import joblib
import os

save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/CICIDS-2017/processed/"

# Save model
joblib.dump(rf_model, save_path + "rf_model.joblib")

# Save results
results = {
    "model": "Random Forest",
    "accuracy": acc,
    "precision": p,
    "recall": r,
    "f1_call": f1,
    "training_time": 1245.48,
    "prediction_time": 13.16
}

import json
with open(save_path + "rf_results.json", "w") as f:
    json.dump(results, f, indent=4)
print("Model saved!")
print("Results saved!")

Model saved!
Results saved!


In [16]:
# Find where files were saved
import os
import subprocess

result = subprocess.run(
    ["find", "/Users/miuyanhong/Desktop", 
     "-name", "rf_model.joblib",
     "-o", "-name", "rf_results.json",
     "-o", "-name", "X_train.npy"],
    capture_output=True,
    text=True
)

print("Files found:")
print(result.stdout)

Files found:
/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/CICIDS-2017/processed/rf_results.json
/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/CICIDS-2017/processed/rf_model.joblib
/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/CICIDS-2017/processed/X_train.npy



In [17]:
# Copy files to new GitHub repo
import shutil
import os

src = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/CICIDS-2017/processed/"

dst = "/Users/miuyanhong/Desktop/TCSS499_Research_Benchmark/data/processed/"

# Copy each file
files = os.listdir(src)
for f in files:
    shutil.copy(src + f, dst + f)
    print(f"Copied: {f}")

print("All files copied!")

Copied: X_clean.npy
Copied: rf_results.json
Copied: rf_model.joblib
Copied: X_test.npy
Copied: X_train.npy
Copied: y_train.csv
Copied: y_test.csv
All files copied!
